# Step 3 — Per-Sample Data Cleaning

## Why does this step exist?

In machine learning, you never train on raw data without asking: *is this data actually telling me what I think it's telling me?*

Single-cell RNA sequencing has the same problem — but the noise sources are specific to how biology works. Raw count matrices from CellRanger contain three categories of systematic contamination that, if left uncorrected, would lead to incorrect biological conclusions. By the end of this step you will have a dataset where each row really is a single live cell, and each value really reflects that cell's own gene expression.

---

### Problem 1 — The soup (ambient RNA)

When you dissociate tissue to get individual cells, some cells rupture. Their RNA spills into the suspension liquid. This free-floating RNA is called the **soup**. Every droplet that gets captured absorbs some of this soup. The result: all your cells appear to express genes they do not actually express. Adipocytes (fat cells) look like they are making hemoglobin. Muscle fibers look like they are making immune proteins. None of that is real.

> **ML analogy:** This is label leakage at the feature level. Your input matrix is contaminated with signal from a different class, distributed across all samples. A model trained on leaky features will appear to work but will generalize to nothing.

**If you do not fix this:** Cell type clusters are blurry. Marker gene signatures are polluted. Differential expression results include false positives from ambient genes rather than real biological changes. The paper's clean separation of MSC states (IPC, CP, CD142+) would be impossible if adipocyte-specific transcripts were bleeding into all other cells.

---

### Problem 2 — Low-quality cells

Not every barcode that passed CellRanger's filter is a healthy, intact cell. You also get:
- **Empty droplets** — a few ambient RNA molecules, no cell
- **Dying or lysing cells** — the membrane is rupturing; cytoplasmic mRNA leaks out, but mitochondria (enclosed organelles) are retained longer. This shifts the RNA profile toward mitochondrial transcripts. A high mitochondrial fraction is the key viability signal.
- **Cell fragments** — pieces of cells, not whole cells

**If you do not fix this:** Dead cells form their own cluster and look like a real cell type. The paper's 22 annotated cell types would include a "dying cell" cluster with no biological meaning.

---

### Problem 3 — Doublets

At 10x's recommended loading density, approximately **3–8% of captured barcodes contain two cells** in one droplet. These doublets appear as a single barcode with roughly twice the gene count. Heterotypic doublets — two cells from *different* types — cluster between their parent populations and are mistaken for rare transitional states.

In this paper, the authors discovered novel MSC sub-populations including a previously unreported Sca1⁻ FAP population in skeletal muscle. A spurious doublet-derived cluster at the edge of the MSC population could have been reported as a discovery. DoubletFinder/Scrublet is the guard against this.

> **ML analogy:** Doublets are mislabeled training examples created by merging two data points into one. The model learns a spurious feature combination that doesn't correspond to any real entity.

---

## Pipeline stages for this study

```
CellRanger output (raw UMI count matrix per sample)
  • Alignment: STAR-based (via CellRanger)
  • Barcode whitelist matching + error correction
  • UMI collapsing within gene (exact/1-edit heuristics)
        │
        ▼
[Stage 1] SoupX ── ambient RNA correction
  • Estimate contamination fraction (ρ):
      – Fixed at ~0.20 (20%) in the paper
      – Typically inferred from low-UMI droplets (empty droplets)
  • Identify “soup profile”:
      – Average expression of genes in empty droplets
  • Adjust counts:
      – Subtract expected ambient contribution per gene per cell
        │
        ▼
[Stage 2] QC filtering ── remove low-quality droplets/cells
  • nFeature_RNA (genes detected per cell):
      – Keep 200–6000
      – Low → empty droplets
      – High → potential doublets
  • nCount_RNA (total UMIs per cell):
      – Keep > 500
  • percent.mt (mitochondrial fraction):
      – Compute: (UMIs from MT genes) / (total UMIs)
      – Keep < 30% (high = stressed/dying cells)
  • Optional additional filters (commonly applied):
      – Ribosomal % thresholds
      – Complexity: log10(nFeature)/log10(nCount)
        │
        ▼
[Stage 2b] Normalization + dimensionality reduction + clustering
  • Normalization:
      – Library size normalize (e.g., scale to 1e4 counts per cell)
      – Log-transform: log1p(counts)
  • Highly variable genes (HVG selection):
      – Variance-stabilizing method (e.g., Seurat v3 flavor)
      – Select ~2,000–3,000 HVGs
  • Scaling:
      – Center and scale genes (z-score)
      – Optionally regress out: nCount, percent.mt
  • PCA:
      – Compute top PCs (typically 30–50)
      – Select PCs via elbow plot / variance explained
  • UMAP:
      – Build kNN graph (k ~ 15–30)
      – UMAP on selected PCs (min_dist ~ 0.3, spread ~ 1.0)
  • Leiden clustering:
      – Graph-based clustering on kNN graph
      – Resolution parameter ~0.5–1.5 controls granularity
        │
        ▼
[Stage 3] Scrublet ── doublet detection
  • Simulate doublets:
      – Combine random pairs of observed transcriptomes
  • Train classifier:
      – Compare observed cells vs simulated doublets in PCA space
  • Score each cell:
      – Doublet score ∈ [0,1]
  • Thresholding:
      – Set cutoff based on bimodal score distribution
      – Expected doublet rate ~3.1% (experiment-dependent)
  • Output:
      – Label cells as singlet or doublet
        │
        ▼
<library_id>_processed.h5ad
  • Filtered, normalized expression matrix
  • Cell metadata: QC metrics, cluster labels, doublet scores

sample_qc_metrics.txt
  • Summary statistics:
      – # cells retained
      – Median genes/UMIs per cell
      – % mitochondrial
      – Doublet rate
```

```
Doublet intuition (how two cells become a “between-cluster” point)

Real biology:
  Cell A expression → x_A
  Cell B expression → x_B
        │
        ▼
Same droplet → sequenced together
        │
        ▼
Observed counts:
  x_obs ≈ x_A + x_B   (raw UMI counts are additive)
        │
        ▼
Normalization (library size scaling + log1p)
  • Removes total count differences
  • Compresses scale
  • Sum behaves like a blended profile
        │
        ▼
Feature selection (HVGs)
  • Keeps genes that distinguish cell types
        │
        ▼
PCA projection
  • Reduces to ~30–50 dimensions
  • Preserves major biological variation
        │
        ▼
Geometry in PCA space:

  Cluster A (cell type A)
        ● ● ●
      ●       ●

                 x  ← doublet (A+B)
              ●

      ●       ●
        ● ● ●
  Cluster B (cell type B)

  • A cells group together
  • B cells group together
  • A+B mixture lands in region influenced by both

        │
        ▼
Doublet detection (e.g., Scrublet)
  • Simulate: x_i + x_j
  • Project into PCA space
  • Compare neighborhoods:
      – Singlets → dense, cluster-consistent
      – Doublets → resemble simulated mixtures

        │
        ▼
Output:
  • Cells flagged if they occupy “mixture-like” regions
```

Across the 39 surviving samples, after all three stages, **204,883 high-quality cells** passed QC.

In [ ]:
import numpy as np
import pandas as pd
import scanpy as sc
import scipy.sparse as sp
import matplotlib.pyplot as plt
import scrublet as scr
from pathlib import Path

## Configuration

These parameters mirror the R function arguments exactly. Review the comments for each before running.

**SoupX parameters:**
- `soupx_b` — whether to run ambient RNA correction at all. Set to `False` only if the tissue dissociation was very clean and you have reason to believe contamination is negligible.
- `soupx_mode` — which contamination estimation strategy to use (see the SoupX section below for details).
- `soupx_fixed_rate` — used only when `soupx_mode` is `"fixed"`.

**Cell filtering parameters:**
- `seurat_filt_b` — whether to apply QC thresholds. Should be `True` in production; set to `False` only for exploratory runs to see unfiltered distributions.
- `nfeature_low/high`, `ncount_low`, `per_mt_high` — thresholds determined from Step 2 diagnostic plots. These are per-sample values; adjust them based on what you observed in the violin and scatter plots.

**PCA / clustering parameters:**
- `pc_num` — number of PCs to use, determined from the Step 2 elbow plot.
- `clust_res` — Leiden resolution. Higher values produce more, smaller clusters. 0.4 is a reasonable starting point; tune after inspecting the UMAP.

**DoubletFinder parameters:**
- `doublet_b` — whether to run doublet detection. Recommended `True` for all 10x experiments.
- `expected_doublet_rate` — 10x Genomics publishes expected doublet rates by cell count: ~3.1% for ~4,000 cells, scaling up with capture. Check the CellRanger metrics from Step 1 and the 10x documentation for the appropriate rate for your cell count.

In [ ]:
cellranger_input_file = "cellranger_manifest.txt"
species       = "mouse"     # "mouse" or "human"
target_folder = Path(".")

# SoupX
soupx_b          = True
soupx_mode       = "auto"   # "auto", "fixed", or "manual"
soupx_fixed_rate = 0.1      # only used when soupx_mode == "fixed"

# Cell filtering
seurat_filt_b = True
nfeature_low  = 200
nfeature_high = 6000
ncount_low    = 400
per_mt_high   = 20.0

# Normalization
norm_mode = "regular"   # "regular" or "sctransform"

# PCA / clustering
pc_num    = 20
clust_res = 0.4

# Doublet detection
doublet_b              = True
expected_doublet_rate  = 0.031   # 3.1% — 10x default for ~4,000 cells

# Output directories
plot_folder  = target_folder / "seurat_plots"
soupx_folder = target_folder / "soupx"
plot_folder.mkdir(exist_ok=True)
if soupx_b:
    soupx_folder.mkdir(exist_ok=True)

mt_prefix = "mt-" if species == "mouse" else "MT-"

## Mitochondrial gene lists

The `manual` SoupX mode requires a set of genes that should be completely absent in most cell types — their presence in a cell is therefore a signature of ambient RNA contamination rather than real expression. The original script uses two gene sets:

- **Hemoglobin genes (`hbGenes`)** — expressed only in red blood cells. When tissue is dissociated, lysed RBCs release hemoglobin transcripts into the soup. These transcripts contaminate all other cell types and would spuriously suggest that adipocytes or muscle cells express hemoglobin.
- **Immunoglobulin heavy-chain genes (`igGenes`)** — expressed only in plasma B cells. Similarly, these can contaminate other cell types and create false signals of immune activity in non-immune tissues.

By identifying cells where these genes are unexpectedly expressed and estimating how much of that expression is ambient, SoupX calibrates its contamination estimate without needing ground-truth doublet labels.

In [ ]:
if species == "mouse":
    hb_genes = ["Hbb-bt", "Hbb-bs", "Hbb-bh2", "Hbb-bh1", "Hbb-y",
                "Hba-x", "Hba-a1", "Hba-a2"]
    ig_genes  = ["Igha", "Ighe", "Ighg2c", "Ighg2b", "Ighg1",
                 "Ighg3", "Ighd", "Ighm", "Ighj4", "Ighj3", "Ighj2", "Ighj1"]
else:
    hb_genes = ["HBB", "HBA1", "HBA2", "HBD", "HBE1", "HBG1", "HBG2", "HBZ"]
    ig_genes  = ["IGHA1", "IGHA2", "IGHE", "IGHG1", "IGHG2",
                 "IGHG3", "IGHG4", "IGHD", "IGHM"]

non_expressing_genes = hb_genes + ig_genes

## Helper — SoupX ambient RNA correction

SoupX (Young & Behjati, 2020, *GigaScience*) corrects for ambient RNA contamination by:

1. **Estimating the soup profile** — CellRanger produces two count matrices: a `filtered_feature_bc_matrix` (cells only) and a `raw_feature_bc_matrix` (all barcodes, including empty droplets). Barcodes in the raw but not the filtered matrix are presumed to be empty droplets filled with ambient RNA. The expression profile of those empty droplets is the "soup" — a sample-specific mixture of ambient transcripts.

2. **Estimating the contamination fraction (rho)** — for each cell, what fraction of its observed counts came from the soup rather than its own transcriptome? SoupX estimates this in three ways:
   - **Auto** — uses the expression patterns of all genes across all cells to estimate rho automatically via an expectation-maximization approach.
   - **Fixed** — applies a fixed contamination fraction (e.g., 10%, 15%, 20%) to all cells. Useful when the auto estimate is unstable or you want to run a sensitivity analysis across contamination assumptions.
   - **Manual** — uses the non-expressing gene sets defined above. Cells that should not express hemoglobin or immunoglobulin genes at all — but do — are used to directly estimate how much of their expression is ambient. This is the most biologically interpretable mode.

3. **Correcting counts** — for each gene in each cell, subtracts the estimated ambient contribution. The result is rounded to integers to maintain count semantics.

> **Note:** The R version saves a separate `.rds` per mode and runs all five modes per sample. The Python implementation runs the selected mode only. To run a sensitivity analysis, loop over multiple `soupx_mode` values.

In [ ]:
def run_soupx(raw_path, filt_path, mode, fixed_rate=0.1, non_expressing_genes=None):
    """
    Ambient RNA correction following the SoupX approach.

    Parameters
    ----------
    raw_path : Path
        Path to CellRanger raw_feature_bc_matrix/ (all barcodes including empties).
    filt_path : Path
        Path to CellRanger filtered_feature_bc_matrix/ (cells only).
    mode : str
        "auto", "fixed", or "manual".
    fixed_rate : float
        Contamination fraction used when mode == "fixed".
    non_expressing_genes : list
        Gene names used to calibrate contamination when mode == "manual".

    Returns
    -------
    AnnData with corrected counts in .X and estimated rho in .obs["soupx_rho"].
    """
    # Load raw (all barcodes) and filtered (cells only) matrices
    adata_raw  = sc.read_10x_mtx(raw_path,  var_names="gene_symbols", cache=False)
    adata_filt = sc.read_10x_mtx(filt_path, var_names="gene_symbols", cache=False)
    adata_raw.var_names_make_unique()
    adata_filt.var_names_make_unique()

    # Identify empty droplets: barcodes in raw but not in filtered.
    # These contain ambient RNA but no cell.
    cell_barcodes  = set(adata_filt.obs_names)
    empty_barcodes = [bc for bc in adata_raw.obs_names if bc not in cell_barcodes]
    adata_empty = adata_raw[empty_barcodes].copy()

    # Estimate the soup (ambient) RNA profile from empty droplets.
    # Each gene's soup fraction is its share of all UMIs in empty droplets.
    soup_counts  = np.array(adata_empty.X.sum(axis=0)).flatten()
    soup_profile = soup_counts / soup_counts.sum()   # gene-wise ambient fraction

    X_filt = adata_filt.X.toarray() if sp.issparse(adata_filt.X) else adata_filt.X.copy()
    cell_totals = X_filt.sum(axis=1, keepdims=True)   # total UMIs per cell

    # Estimate contamination fraction rho per cell
    if mode == "auto":
        # Auto: estimate rho as the fraction of each cell's counts that
        # is consistent with the soup profile. For each cell, the
        # expected soup contribution is rho * total_counts * soup_profile.
        # We solve for the rho that minimizes the residual between
        # observed and expected counts for non-highly-expressed genes.
        # Simplified approach: use the median ratio of observed-to-expected
        # ambient counts across genes expressed at low levels.
        expected_soup = cell_totals * soup_profile[np.newaxis, :]  # cells x genes
        # Genes where soup is the dominant signal (low mean, high soup fraction)
        low_expr_mask = X_filt.mean(axis=0) < np.percentile(X_filt.mean(axis=0), 25)
        with np.errstate(divide="ignore", invalid="ignore"):
            rho_per_gene = np.where(
                expected_soup[:, low_expr_mask] > 0,
                X_filt[:, low_expr_mask] / expected_soup[:, low_expr_mask],
                np.nan
            )
        rho_cells = np.nanmedian(rho_per_gene, axis=1)
        rho_cells = np.clip(rho_cells, 0, 0.5)   # cap at 50%
        print(f"  Auto rho estimate: {rho_cells.mean():.3f} (mean across cells)")

    elif mode == "fixed":
        rho_cells = np.full(X_filt.shape[0], fixed_rate)
        print(f"  Fixed contamination rate: {fixed_rate}")

    elif mode == "manual":
        # Manual: use only the non-expressing gene sets to calibrate rho.
        # Cells should not express hemoglobin or immunoglobulin genes.
        # Any counts observed in those genes in non-RBC / non-plasma cells
        # must be ambient. We use this to directly estimate rho.
        marker_idx = [i for i, g in enumerate(adata_filt.var_names)
                      if g in non_expressing_genes]
        if len(marker_idx) == 0:
            raise ValueError("None of the non-expressing marker genes found in the dataset. "
                             "Check gene names match the reference used by CellRanger.")
        X_markers    = X_filt[:, marker_idx]
        soup_markers = soup_profile[marker_idx]
        expected_markers = cell_totals * soup_markers[np.newaxis, :]
        with np.errstate(divide="ignore", invalid="ignore"):
            rho_per_marker = np.where(
                expected_markers > 0,
                X_markers / expected_markers,
                np.nan
            )
        rho_cells = np.nanmedian(rho_per_marker, axis=1)
        rho_cells = np.clip(rho_cells, 0, 0.5)
        print(f"  Manual rho estimate using {len(marker_idx)} marker genes: "
              f"{rho_cells.mean():.3f} (mean)")

    # Adjust counts: subtract estimated ambient contribution per cell.
    # Expected ambient counts for cell i, gene j = rho_i * total_i * soup_j.
    # Round to integers to preserve count semantics (as in the R version).
    expected_ambient = rho_cells[:, np.newaxis] * cell_totals * soup_profile[np.newaxis, :]
    X_corrected = np.round(np.maximum(X_filt - expected_ambient, 0)).astype(np.float32)

    adata_out = adata_filt.copy()
    adata_out.X = sp.csr_matrix(X_corrected)
    adata_out.obs["soupx_rho"] = rho_cells

    return adata_out

## Helper — Doublet detection with Scrublet

DoubletFinder (McGinnis et al., 2019, *Cell Systems*) and Scrublet (Wolock et al., 2019, *Cell Systems*) both detect doublets by simulating artificial doublets in silico and asking whether real barcodes look more like singlets or doublets in PCA space.

**How it works:**
1. A set of synthetic doublets is created by randomly summing pairs of real cells' count vectors — mimicking what happens when two cells are captured in one droplet.
2. PCA is computed jointly on real cells and synthetic doublets.
3. A doublet score is assigned to each real barcode based on how many of its nearest neighbors in PCA space are synthetic doublets. A barcode surrounded by synthetic doublets is likely a real doublet.

**Why doublets matter:**
- At 10x's recommended loading density of ~4,000–8,000 cells, approximately 3–8% of captured barcodes are doublets. This scales with cell count: ~1.6% per 1,000 cells loaded.
- Doublets that consist of two different cell types appear as hybrid transcriptomes and cluster between the two parent populations. They can be mistaken for rare transitional cell states or novel cell types — a significant source of biological misinterpretation.
- Homotypic doublets (two cells of the same type) are harder to detect and less damaging; they primarily inflate cluster size rather than creating spurious clusters.

**Homotypic correction:**
The expected doublet rate is not purely heterotypic. The fraction of doublets that are homotypic depends on the relative proportions of each cell type. DoubletFinder corrects the expected count using the cluster composition (`modelHomotypic`). Scrublet handles this similarly via the `expected_doublet_rate` parameter, which sets the prior for the simulation.

In [ ]:
def run_doublet_detection(adata, expected_doublet_rate=0.031, random_state=42):
    """
    Score and classify doublets using Scrublet.

    Scrublet simulates synthetic doublets from the observed count matrix and
    scores each real barcode by how doublet-like it is in PCA space.

    Parameters
    ----------
    adata : AnnData
        Object with raw counts in .X.
    expected_doublet_rate : float
        Prior on the fraction of barcodes that are doublets. Use the 10x
        expected rate for your cell count (0.031 for ~4,000 cells).

    Returns
    -------
    AnnData with doublet_score and predicted_doublet columns added to .obs.
    """
    counts_matrix = adata.X.toarray() if sp.issparse(adata.X) else adata.X

    scrub = scr.Scrublet(
        counts_matrix,
        expected_doublet_rate=expected_doublet_rate,
        random_state=random_state
    )
    doublet_scores, predicted_doublets = scrub.scrub_doublets(verbose=False)

    adata.obs["doublet_score"]     = doublet_scores
    adata.obs["predicted_doublet"] = predicted_doublets

    n_doublets  = predicted_doublets.sum()
    n_singlets  = (~predicted_doublets).sum()
    print(f"  Doublets predicted: {n_doublets} ({100*n_doublets/len(predicted_doublets):.1f}%)")
    print(f"  Singlets retained:  {n_singlets}")

    return adata

## Load manifest

In [ ]:
# The manifest has no header in the R version; columns are positional:
# col 0 = project_folder_path, col 1 = flowcell_ID, col 2 = library_ID
manifest = pd.read_csv(
    cellranger_input_file, sep="\t",
    header=None, names=["project_folder_path", "flowcell_ID", "library_ID"]
)
print(f"Loaded manifest with {len(manifest)} samples")
manifest.head()

## Per-sample processing loop

Each sample goes through the full pipeline: SoupX → QC filter → normalize → PCA → UMAP → cluster → doublet detection. QC statistics are recorded at each stage to track cell attrition.

In [ ]:
sample_stats = []

for _, row in manifest.iterrows():
    lib_id   = row["library_ID"]
    outs_dir = Path(row["project_folder_path"]) / row["flowcell_ID"] / lib_id / "outs"
    filt_dir = outs_dir / "filtered_feature_bc_matrix"
    raw_dir  = outs_dir / "raw_feature_bc_matrix"

    print(f"\n{'='*60}")
    print(f"Sample: {lib_id}")
    print(f"{'='*60}")

    stats = {"library_id": lib_id}

    # ================================================================
    # Stage 1 — SoupX ambient RNA decontamination
    # ================================================================
    if soupx_b:
        print(f"\n[1] SoupX ({soupx_mode} mode)")
        adata = run_soupx(
            raw_path=raw_dir,
            filt_path=filt_dir,
            mode=soupx_mode,
            fixed_rate=soupx_fixed_rate,
            non_expressing_genes=non_expressing_genes,
        )
        stats["soupx_mode"]  = soupx_mode
        stats["soupx_rho"]   = float(adata.obs["soupx_rho"].mean())
    else:
        print("[1] SoupX skipped — loading filtered matrix directly")
        adata = sc.read_10x_mtx(filt_dir, var_names="gene_symbols", cache=False)
        adata.var_names_make_unique()

    # Prefix barcodes with library ID to keep them unique after merging
    adata.obs_names = [f"{lib_id}_{bc}" for bc in adata.obs_names]
    stats["cell_num_after_cellranger"] = adata.n_obs
    print(f"  Cells loaded: {adata.n_obs:,}")

    # ================================================================
    # Stage 2 — QC metric calculation and cell filtering
    # ================================================================
    print("\n[2] Seurat-style QC filtering")

    # Mitochondrial gene fraction:
    # High percent_mt indicates a dying or lysed cell. When a cell begins
    # to lyse, cytoplasmic mRNA escapes through the compromised membrane,
    # but mitochondria (and their enclosed RNA) are retained longer.
    # This shifts the RNA profile toward mitochondrial transcripts,
    # making percent_mt a sensitive viability indicator.
    adata.var["mt"] = adata.var_names.str.startswith(mt_prefix)
    sc.pp.calculate_qc_metrics(
        adata, qc_vars=["mt"], percent_top=None, log1p=False, inplace=True
    )

    if seurat_filt_b:
        stats.update({
            "nFeature_RNA_low":  nfeature_low,
            "nFeature_RNA_high": nfeature_high,
            "nCount_RNA_low":    ncount_low,
            "percent_MT_high":   per_mt_high,
        })
        # Apply all three thresholds simultaneously.
        # Cells failing any threshold are removed.
        keep = (
            (adata.obs["n_genes_by_counts"]  > nfeature_low)  &
            (adata.obs["n_genes_by_counts"]  < nfeature_high) &
            (adata.obs["total_counts"]        > ncount_low)    &
            (adata.obs["pct_counts_mt"]       < per_mt_high)
        )
        adata = adata[keep].copy()
        stats["cell_num_after_seurat_filt"] = adata.n_obs
        print(f"  Cells after QC filter: {adata.n_obs:,} "
              f"(removed {keep.sum() - adata.n_obs + (keep.shape[0] - keep.sum()):,})")
    else:
        print("  Cell filtering skipped")

    # Post-filter violin and scatter plots
    # These show the distributions AFTER filtering — useful for confirming
    # that the thresholds removed the low-quality tail without cutting
    # into the main population.
    fig, axes = plt.subplots(1, 3, figsize=(9, 4))
    fig.suptitle(f"{lib_id} (post-filter)", fontsize=11)
    for ax, (col, label) in zip(axes, [
        ("n_genes_by_counts", "nFeature_RNA"),
        ("total_counts",      "nCount_RNA"),
        ("pct_counts_mt",     "percent.mt"),
    ]):
        ax.violinplot(adata.obs[col], positions=[0], showmedians=True)
        ax.set_xticks([])
        ax.set_ylabel(label)
    plt.tight_layout()
    fig.savefig(plot_folder / f"{lib_id}_violin.png", dpi=150)
    plt.show(); plt.close(fig)

    fig, axes = plt.subplots(1, 2, figsize=(9, 4))
    fig.suptitle(f"{lib_id} (post-filter)", fontsize=11)
    axes[0].scatter(adata.obs["total_counts"], adata.obs["pct_counts_mt"],
                    s=1, alpha=0.3, rasterized=True)
    axes[0].set_xlabel("nCount_RNA"); axes[0].set_ylabel("percent.mt")
    axes[1].scatter(adata.obs["total_counts"], adata.obs["n_genes_by_counts"],
                    s=1, alpha=0.3, rasterized=True)
    axes[1].set_xlabel("nCount_RNA"); axes[1].set_ylabel("nFeature_RNA")
    plt.tight_layout()
    fig.savefig(plot_folder / f"{lib_id}_scatter.png", dpi=150)
    plt.show(); plt.close(fig)

    # ================================================================
    # Stage 2b — Normalization and feature selection
    # ================================================================
    stats["normalization_method"] = norm_mode

    # Store raw counts before normalization — needed by DoubletFinder
    # and required by downstream tools that expect integer counts in a layer.
    adata.layers["counts"] = adata.X.copy()

    if norm_mode == "regular":
        # Normalize each cell to the per-sample median count, then log1p.
        # Using the per-sample median (rather than a fixed 10,000) makes
        # normalized values more meaningful when samples differ substantially
        # in sequencing depth.
        target_sum = float(np.median(adata.obs["total_counts"]))
        sc.pp.normalize_total(adata, target_sum=target_sum)
        sc.pp.log1p(adata)
        sc.pp.scale(adata, max_value=10)
    elif norm_mode == "sctransform":
        # SCTransform regresses out sequencing depth via a regularized
        # negative binomial model, producing Pearson residuals that are
        # depth-independent. The scanpy approximation below (normalize
        # → log1p → scale) captures the intent but lacks the full
        # regularization. For a faithful SCTransform port, use rpy2.
        sc.pp.normalize_total(adata)
        sc.pp.log1p(adata)
        sc.pp.scale(adata, max_value=10)

    # Highly variable genes: identify the ~2,000 genes with the highest
    # biological variance relative to their mean. PCA on all ~20,000 genes
    # would be dominated by technical noise; restricting to variable genes
    # focuses the dimensionality reduction on genes that actually
    # distinguish cell types.
    sc.pp.highly_variable_genes(adata, n_top_genes=2000)

    # ================================================================
    # Stage 2c — Dimensionality reduction, UMAP, clustering
    # ================================================================

    # PCA: linear dimensionality reduction. We use only the highly
    # variable genes to keep the decomposition biologically meaningful.
    sc.tl.pca(adata, n_comps=50, use_highly_variable=True)

    # Elbow plot: variance explained per PC. The elbow guides the choice
    # of pc_num, which controls how many PCs feed into the neighbor
    # graph, UMAP, and doublet detection.
    variance_ratio = adata.uns["pca"]["variance_ratio"]
    fig, ax = plt.subplots(figsize=(5, 4))
    ax.plot(range(1, len(variance_ratio) + 1), variance_ratio, "o-", ms=4)
    ax.axvline(pc_num, color="red", linestyle="--", alpha=0.6, label=f"pc_num={pc_num}")
    ax.set_xlabel("PC"); ax.set_ylabel("Fraction of variance explained")
    ax.set_title(f"{lib_id} — PCA elbow")
    ax.legend()
    plt.tight_layout()
    fig.savefig(plot_folder / f"{lib_id}_elbow.png", dpi=150)
    plt.show(); plt.close(fig)

    # Neighbor graph: connects each cell to its k nearest neighbors in
    # the top pc_num-dimensional PCA space. This graph is the foundation
    # for both UMAP layout and Leiden clustering.
    sc.pp.neighbors(adata, n_pcs=pc_num)

    # UMAP: non-linear 2D projection for visualization. UMAP preserves
    # local neighborhood structure — cells that are transcriptionally
    # similar appear close together. Note that UMAP distances are NOT
    # quantitatively meaningful; only topology (which clusters are
    # connected or separated) should be interpreted.
    sc.tl.umap(adata)

    # Leiden clustering: community detection on the neighbor graph.
    # Resolution controls granularity — higher values produce more clusters.
    # These clusters are used by DoubletFinder to estimate homotypic
    # doublet proportions; final cell type annotation happens in Step 7.
    sc.tl.leiden(adata, resolution=clust_res, key_added="leiden")
    stats["number_of_PCs_used"]    = pc_num
    stats["clustering_resolution"] = clust_res
    print(f"  Leiden clusters: {adata.obs['leiden'].nunique()}")

    # ================================================================
    # Stage 3 — Doublet detection
    # ================================================================
    if doublet_b:
        print("\n[3] Doublet detection (Scrublet)")
        # Scrublet expects raw (not normalized) counts.
        # We use the counts layer saved before normalization.
        adata_raw_counts = adata.copy()
        adata_raw_counts.X = adata.layers["counts"]
        adata_raw_counts = run_doublet_detection(
            adata_raw_counts,
            expected_doublet_rate=expected_doublet_rate
        )
        adata.obs["doublet_score"]     = adata_raw_counts.obs["doublet_score"]
        adata.obs["predicted_doublet"] = adata_raw_counts.obs["predicted_doublet"]

        # Tag cells — "Doublet" / "Singlet" to match R output column name
        adata.obs["Doublet"] = adata.obs["predicted_doublet"].map(
            {True: "Doublet", False: "Singlet"}
        )
        n_singlets = (adata.obs["Doublet"] == "Singlet").sum()
        stats["cell_num_after_df"] = int(n_singlets)
    else:
        print("[3] Doublet detection skipped")

    # ================================================================
    # Save per-sample object
    # ================================================================
    out_path = target_folder / f"{lib_id}_processed.h5ad"
    adata.write_h5ad(out_path)
    print(f"\nSaved: {out_path}")

    sample_stats.append(stats)

print("\nAll samples complete.")

## Write QC metrics table

The `sample_qc_metrics.txt` table tracks cell counts at each stage of the pipeline for every sample. This provides:

- **Audit trail** — a record of how many cells were removed at each stage and why, making results reproducible and reviewable.
- **QC flag detection** — if one sample loses an unusually high fraction of cells at a particular stage (e.g., 60% of cells removed by the mt filter vs. 10% in other samples), it may indicate a sample-specific dissociation or library prep issue worth investigating before integration.
- **Methods reporting** — cell counts at each stage are typically reported in the methods section of publications.

In [ ]:
stats_df = pd.DataFrame(sample_stats)
out_path = target_folder / "sample_qc_metrics.txt"
stats_df.to_csv(out_path, sep="\t", index=False)
print(f"Written: {out_path}")
stats_df

## Next steps

The per-sample `.h5ad` files produced here feed into Step 4 (pseudobulk clustering sanity check) and Step 5 (multi-sample integration). Before proceeding:

1. **Review the QC metrics table** — flag any samples with atypical cell attrition rates.
2. **Inspect the post-filter violin plots** — confirm the low-quality tail has been removed and the remaining distribution is unimodal.
3. **Check the doublet rates** — the predicted doublet fraction should be in the expected range for your cell count (see the 10x documentation). Unusually high doublet fractions may indicate overloading or cell clumping during dissociation.
4. **Decide whether to remove doublets now or in Step 5** — the `.h5ad` files retain all cells with doublet labels. You can filter on `Doublet == "Singlet"` here, or carry the labels into the integrated object and remove doublets after integration (which can be more accurate since UMAP context helps confirm doublet calls).